
# DL Assignment 03

**Name:** Abdul Aziz

**Course Email:** tamimaziz2007@gmail.com


## End of Assignment

Before submitting:
- Run all cells from top to bottom.  
- Check that all answer sections are filled.  
- Instruction video অনুযায়ী আমাদের দেয়া Colab ফাইলটি থেকে প্রথম একটি Save copy in drive করে নিবা। এরপর Google colab এর মধ্যে কোডগুলো করবে এবং সেই ফাইলটি ‘Anyone with the link’ & ‘View’ Access দিয়ে ফাইলটির Shareble Link টি সাবমিট করবে।

# General Instruction

You must choose your own dataset.

The dataset must:

Be a supervised learning dataset (Regression or Binary Classification)

Contain at least 300 samples

Have at least 2 input features

Be in CSV format

You are NOT allowed to use Dataset or DataLoader.

You must implement everything manually.

# Question 01: [ Marks 05 ]

## Dataset Preparation

## Using your chosen dataset:

Load the dataset.

Perform necessary preprocessing:

Handle missing values (if any)

Encode categorical variables (if necessary)

Feature scaling (if needed)

Separate features (X) and target (y).

Convert them into NumPy arrays.

Convert them into PyTorch tensors.

Split into training and testing sets.

Clearly explain each preprocessing decision.

# **Write** Answer 01:


In [13]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch.optim as optim

df = pd.read_csv("diabetes.csv")
df.head()
df.shape

col_invalid_zero = ['Glucose', 'BloodPressure', 'SkinThickness','Insulin','BMI']
df[col_invalid_zero] = df[col_invalid_zero].replace(0, np.nan)

df[col_invalid_zero] = df[col_invalid_zero].fillna(df[col_invalid_zero].mean())
X = df.drop('Outcome', axis=1)
y = df['Outcome']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_np = np.array(X_scaled)
y_np = np.array(y)

X_train, X_test, y_train, y_test = train_test_split(X_np, y_np, test_size=0.2, random_state=42)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1,1)

print("Train shape:", X_train_tensor.shape)
print("Test shape:", X_test_tensor.shape)

Train shape: torch.Size([614, 8])
Test shape: torch.Size([154, 8])


# Question 02: [ Marks 20 ]

## Design a neural network using nn.Module.

### The model must contain:

Input layer

At least one hidden layer

Output layer

Suitable activation function



## Justify:

Number of hidden neurons

Choice of activation function

Print  the total number of trainable parameters.


## Write Answer 02:


In [14]:
class DiabetesNet(nn.Module):
  def __init__(self, input_size):
    super(DiabetesNet, self).__init__()
    self.hidden1 = nn.Linear(input_size, 16)
    self.relu1 = nn.ReLU()

    self.hidden2 = nn.Linear(16, 8)
    self.relu2 = nn.ReLU()

    self.output = nn.Linear(8,1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    x = self.relu1(self.hidden1(x))
    x = self.relu2(self.hidden2(x))
    x = self.sigmoid(self.output(x))
    return x

input_size = X_train_tensor.shape[1]
model = DiabetesNet(input_size)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total trainable parameter:", total_params)


Total trainable parameter: 289


# Question 03: [ Marks 10 ]

Choose an appropriate loss function.

Choose an optimizer.

<br>

Justify your choices based on:

Regression vs Classification

Nature of the dataset

## Write Answer 03:

In [15]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Question 04: [ Marks 15 ]

## Implement a full training loop:

Forward pass

Loss computation

Backward pass

Parameter update

Gradient reset

### Requirements:

Train for at least 100 epochs.

Print loss every 10 epochs.

Store training loss history(You can pick your own Data Structure).

Explain clearly what happens in each step of the pipeline.

## Write Answer 04:

In [16]:
num_epochs = 100
loss_history = []

for epoch in range(num_epochs):
  model.train()
  y_pred = model(X_train_tensor)
  loss = criterion(y_pred, y_train_tensor)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()
  loss_history.append(loss.item())
  if(epoch+1) % 10 == 0:
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

Epoch [10/100], Loss: 0.6688
Epoch [20/100], Loss: 0.6571
Epoch [30/100], Loss: 0.6470
Epoch [40/100], Loss: 0.6372
Epoch [50/100], Loss: 0.6265
Epoch [60/100], Loss: 0.6144
Epoch [70/100], Loss: 0.6011
Epoch [80/100], Loss: 0.5866
Epoch [90/100], Loss: 0.5718
Epoch [100/100], Loss: 0.5570


# Question 05: [ Marks 10 ]

## Evaluate the model on test data.

## For regression:

Report MSE and MAE


## For classification:

Report Accuracy

Compare training vs testing performance.

State whether the model is underfitting or overfitting.

## Write Answer 05:

In [17]:
model.eval()
with torch.no_grad():
  train_pred = model(X_train_tensor)
  train_pred_labels = (train_pred >= 0.5).float()
  train_accuracy = (train_pred_labels == y_train_tensor).float().mean()

  test_pred = model(X_test_tensor)
  test_pred_labels = (test_pred >= 0.5).float()
  test_accuracy = (test_pred_labels == y_test_tensor).float().mean()

print(f"Training Accuracy: {train_accuracy.item()*100:.2f}%")
print(f"Testing Accuracy: {test_accuracy.item()*100:.2f}%")

Training Accuracy: 76.22%
Testing Accuracy: 73.38%


# Question 06: [ Marks 20 ]

## Modify at least ONE of the following:

Learning rate

Number of hidden neurons

Number of epochs

### Train again and compare:

Convergence speed

Final performance

Explain how the change affected the model.

## Write Answer 06:

In [18]:
model2 = DiabetesNet(input_size)
criterion2 = nn.BCELoss()
optimizer2 = optim.Adam(model2.parameters(), lr=0.01)

num_epochs = 100
loss_history2 = []

for epoch in range(num_epochs):
  model2.train()
  y_pred2 = model2(X_train_tensor)
  loss2 = criterion2(y_pred2, y_train_tensor)

  optimizer2.zero_grad()
  loss2.backward()
  optimizer2.step()

  loss_history2.append(loss2.item())

  if(epoch+1)%10 == 0:
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss2.item():.4f}")

model2.eval()
with torch.no_grad():
  train_pred2 = model2(X_train_tensor)
  train_pred_labels2 = (train_pred2 >= 0.5).float()
  train_accuracy2 = (train_pred_labels2 == y_train_tensor).float().mean()

  test_pred2 = model2(X_test_tensor)
  test_pred_labels2 = (test_pred2 >= 0.5).float()
  test_accuracy2 = (test_pred_labels2 == y_test_tensor).float().mean()

print(f"New Training Accuracy: {train_accuracy2.item()*100:.2f}%")
print(f"New Testing Accuracy: {test_accuracy2.item()*100:.2f}%")

Epoch [10/100], Loss: 0.5653
Epoch [20/100], Loss: 0.4805
Epoch [30/100], Loss: 0.4536
Epoch [40/100], Loss: 0.4376
Epoch [50/100], Loss: 0.4242
Epoch [60/100], Loss: 0.4118
Epoch [70/100], Loss: 0.4008
Epoch [80/100], Loss: 0.3906
Epoch [90/100], Loss: 0.3802
Epoch [100/100], Loss: 0.3688
New Training Accuracy: 82.08%
New Testing Accuracy: 73.38%


# Question 07: [ Marks 20 ]


# Training Analysis

Answer the following:

Why must gradients be reset every epoch?

What happens if learning rate is too high?

What happens if learning rate is too small?

Why do we define layers inside the constructor (__init__) and not inside forward()?


## Write Answer 07:

Answer 1:-

PyTorch accumulates gradients by default it adds new gradients to old ones instead of replacing them. If we don't call `optimizer.zero_grad()`, gradients from previous epochs would mix with the current one, leading to wrong updates. Resetting ensures each epoch's update is based only on that epoch's actual error.

Answer 2:-

The model takes very large steps while updating weights, often overshooting the optimal point instead of setting into it. This usually causes the loss to fluctuate up and down instead of decreasing smoothly, or in bad cases the loss can blow up completely. Training becomes unstable and may never converge properly.

Answer 3:-

The model updates weights in very tiny steps, so training becomes extremely slow. The loss does decrease, but very gradually and if the number of epochs isn't high enough the model may not reach a good solution at all. It's more stable, but inefficient and time consuming.

Answer 4:-

Layers like `nn.Linear` hold trainable parameters (weights and biases) that must be created once and reused across every forward pass. If defined inside `forward()` new random layers would be created every time, so the model would never retain what it learned. That's why layers are created once in `__init__` and `forward()` simply defines how data flows through those already-existing layers.